<a href="https://colab.research.google.com/github/Netrahoni/FlyRankAi-Intern-work-Files/blob/main/work/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Content Opportunity & Refresh Scoring Engine**
* **Internship Track:** Refresh / Content Opportunity Scoring
* **Author:** Netra Mani Pokhrel ([Netrahoni](https://github.com/Netrahoni))
* **Objective:** Build an honest machine learning pipeline using DuckDB and Scikit-Learn to identify decaying web pages and generate a prioritized editorial refresh playbook.

In [ ]:
# ==========================================
# 1. ENVIRONMENT SETUP & AUTHENTICATION
# ==========================================
# Install DuckDB for high-performance querying over Parquet files
!pip install duckdb -q

import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Securely retrieve your Hugging Face read token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    # Fallback input prompt if secrets manager is bypassed
    import getpass
    hf_token = getpass.getpass("Enter your Hugging Face Token: ")

# Initialize DuckDB and establish Hugging Face authentication
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

print("✅ Setup complete! DuckDB is authenticated and ready.")

## **Step 2: Data Extraction & Feature Engineering**
We query the FlyRank warehouse release (`FlyRank/internship-warehouse`) using DuckDB.
* **Historical Feature Window:** November 1, 2025 – April 30, 2026 (6 months of baseline data).
* **Future Label Window:** May 1, 2026 – May 31, 2026 (1 month forward window).
* **Target Label (`needs_refresh`):** Binary flag set to `1` if future clicks dropped below 85% of the historical monthly average, signaling active content decay.

In [ ]:
# ==========================================
# 2. SQL DATA EXTRACTION VIA DUCKDB
# ==========================================
query = """
WITH historical_features AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as total_clicks,
        SUM(gsc_impressions) as total_impressions,
        AVG(gsc_avg_position) as avg_position,
        (SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0)) as ctr
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2025-11-01' AND '2026-04-30'
    GROUP BY content_hash_id
),
future_labels AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as future_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-05-01' AND '2026-05-31'
    GROUP BY content_hash_id
)
SELECT
    h.*,
    f.future_clicks,
    CASE WHEN f.future_clicks < (h.total_clicks / 6.0) * 0.85 THEN 1 ELSE 0 END as needs_refresh
FROM historical_features h
JOIN future_labels f ON h.content_hash_id = f.content_hash_id
WHERE h.total_impressions > 1000
"""

df = con.execute(query).df()
print(f"🚀 Success! Downloaded {df.shape[0]:,} rows and {df.shape[1]} columns of data.")
df.head()

## **Step 3: Model Training & Time-Aware Validation**
To prevent data leakage, we utilize a strict chronological split (80% training, 20% testing). We compare a simple **CTR Heuristic Baseline** against a robust **Random Forest Classifier** to ensure our model adds genuine predictive value over basic guessing.

In [ ]:
# ==========================================
# 3. MACHINE LEARNING & EVALUATION
# ==========================================
# Clean missing values
df = df.fillna(0)

# Define features and labels
features = ['total_clicks', 'total_impressions', 'avg_position', 'ctr']
X = df[features]
y = df['needs_refresh']

# Time-aware split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline Evaluation (Heuristic: Low CTR = Decay)
average_ctr = X_train['ctr'].mean()
baseline_preds = (X_test['ctr'] < average_ctr).astype(int)

# Random Forest Model
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

print("--- BASELINE SCORECARD ---")
print(classification_report(y_test, baseline_preds, zero_division=0))

print("\n--- RANDOM FOREST MODEL SCORECARD ---")
print(classification_report(y_test, model_preds, zero_division=0))

##  **Step 4: The Ranked Action Playbook**
We convert model probabilities into business impact by calculating an **Opportunity Score**:
$$\text{Opportunity Score} = \text{Refresh Probability} \times \text{Total Impressions}$$
This guarantees that high-traffic pages at high risk of decay bubble straight to the top of the editorial team's queue.

In [ ]:
# ==========================================
# 4. GENERATING THE ACTION PLAYBOOK
# ==========================================
probabilities = model.predict_proba(X_test)[:, 1]

results = X_test.copy()
results['content_hash_id'] = df.loc[X_test.index, 'content_hash_id']
results['refresh_probability'] = probabilities
results['opportunity_score'] = results['refresh_probability'] * results['total_impressions']

action_playbook = results.sort_values(by='opportunity_score', ascending=False)

print("🏆 TOP 10 CONTENT REFRESH TARGETS:")
action_playbook[['content_hash_id', 'total_impressions', 'refresh_probability', 'opportunity_score']].head(10)